In [26]:
!pip install langchain langchain-community langgraph chromadb pypdf sentence-transformers transformers accelerate
!pip install --upgrade transformers accelerate


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Load PDF + Chunking

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("customer_support.pdf")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

print("Chunks:", len(chunks))

C:\Users\siddharth\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunks: 3


## Free Embeddings + Store (ChromaDB)

In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever()

C:\Users\siddharth\AppData\Local\Temp\ipykernel_22072\3237127633.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4608.39it/s]


## LLM Answer Generation

In [22]:
from transformers import pipeline

llm_pipeline = pipeline(
    "text-generation",
    model="gpt2"
)

def generate_answer(query, docs):
    context = "\n".join([d.page_content for d in docs[:2]])

    prompt = f"""
Context:
{context}

Question:
{query}

Answer:
"""

    result = llm_pipeline(
        prompt,
        max_new_tokens=80,
        do_sample=False
    )

    output = result[0]["generated_text"]

    # remove prompt part
    return output.replace(prompt, "").strip()

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████| 148/148 [00:00<00:00, 4466.97it/s]


## Decision logic (Routing)

In [23]:
def check_escalation(answer, docs):
    # Only escalate if NO documents found
    if len(docs) == 0:
        return "escalate"
    
    # Otherwise ALWAYS respond
    return "respond"

## LangGraph Workflow

In [24]:
from langgraph.graph import StateGraph

from typing import TypedDict, List

class State(TypedDict):
    query: str
    docs: List
    answer: str
    decision: str

def process_node(state):
    docs = retriever.invoke(state["query"])
    answer = generate_answer(state["query"], docs)

    state["docs"] = docs
    state["answer"] = answer

    return state


def decision_node(state):
    decision = check_escalation(state["answer"], state["docs"])
    state["decision"] = decision
    return state


graph = StateGraph(State)

graph.add_node("process", process_node)
graph.add_node("decision", decision_node)

graph.set_entry_point("process")
graph.add_edge("process", "decision")
graph.set_finish_point("decision")

app = graph.compile()

## HITL (Human-in-the-Loop)

In [25]:
def human_response(query):
    return f"[HUMAN SUPPORT] Your query '{query}' has been escalated."

## Running the system

In [31]:
query = "How do I track my order?"

result = app.invoke({"query": query})

if result["decision"] == "escalate":
    print(human_response(query))
else:
    print(result["answer"])

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1. Order Tracking

Users can track their orders using the tracking ID available in their account dashboard.

2. Payment Issues

If a payment fails, users should retry using a different method or contact their bank. If the issue

persists, contact support.

7. Technical Issues

For app crashes or bugs, users should restart the application or rein
